# Study 1 - Space

**Figure 1 (Combinatoric Space): Human vs Machine divergent thinking. Latest data (45-model midpoint machine + 12,147 human).**

- DAT scored with the Olson (2021) GloVe scorer (mean pairwise cosine distance x100).
- Palette: Human = purple (#5E348B), Machine = teal (#3CB7B0). Scatter outlined, darkened 20%, alpha 0.36.
- 1:1 plot aspect. Gray grids: Panel A horizontal only, Panel B both axes.

## Panel A - Divergent Thinking Score by Split
Bottom 10% / Middle 80% / Top 10%; n=500/group sampled; Welch t stars + mean diff (Machine-Human).

## Panel B - Cumulative Distinct Words
Top 10%, first 7 valid words; both sides capped at 500 sampled responses; between-respondent bootstrap 95% CI.


## Panel A code

In [ ]:
import csv, numpy as np, random
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def sc(rows,k):
    o=[]
    for r in rows:
        try:o.append(float(r.get(k,'')))
        except:pass
    return np.array(o)
Hs=sc(H,'word_dat_score'); Ms=sc(M,'dat_score')
def splits(ss):
    lo,hi=np.percentile(ss,10),np.percentile(ss,90)
    return [ss[ss<=lo], ss[(ss>lo)&(ss<hi)], ss[ss>=hi]]
Hsp=splits(Hs); Msp=splits(Ms)
# SAMPLE: fixed n per group per side (not whole population)
SAMPLE_N=500
def samp(a): return np.random.choice(a, min(SAMPLE_N,len(a)), replace=False)
Hsp=[samp(a) for a in Hsp]; Msp=[samp(a) for a in Msp]
HUMAN="#5E348B"; MACHINE="#3CB7B0"
def darken(hexc,f=0.64):
    hexc=hexc.lstrip('#'); r,g,b=[int(hexc[i:i+2],16) for i in (0,2,4)]
    return (r*f/255,g*f/255,b*f/255)
HUMAN_D=darken(HUMAN); MACHINE_D=darken(MACHINE)
groups=["Bottom 10%","Middle 80%","Top 10%"]
def ci95(a): return 1.96*np.std(a,ddof=1)/np.sqrt(len(a))
def stars(p): return "***" if p<1e-3 else "**" if p<1e-2 else "*" if p<0.05 else "ns"
ANNOT_SIZE=10; ANNOT_COLOR="#333333"
fig,ax=plt.subplots(figsize=(7,7))
x=np.arange(3); w=0.36
for i in range(3):
    h=Hsp[i]; m=Msp[i]
    ax.bar(i-w/2, h.mean(), w, color=HUMAN, alpha=0.85, zorder=2)
    ax.bar(i+w/2, m.mean(), w, color=MACHINE, alpha=0.85, zorder=2)
    def jit(vals,center):
        return center+np.random.uniform(-w/2.6,w/2.6,len(vals)), vals
    jx,jv=jit(h,i-w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=HUMAN_D,alpha=0.36,linewidths=0.6,zorder=3)
    jx,jv=jit(m,i+w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=MACHINE_D,alpha=0.36,linewidths=0.6,zorder=3)
    ax.errorbar(i-w/2,h.mean(),yerr=ci95(h),color='black',capsize=4,lw=1.4,zorder=5)
    ax.errorbar(i+w/2,m.mean(),yerr=ci95(m),color='black',capsize=4,lw=1.4,zorder=5)
    t,p=stats.ttest_ind(h,m,equal_var=False)
    diff=m.mean()-h.mean()
    ytop=max(h.mean(),m.mean())+9
    ax.plot([i-w/2,i-w/2,i+w/2,i+w/2],[ytop-1.5,ytop,ytop,ytop-1.5],color='black',lw=1.1,zorder=5)
    ax.text(i,ytop+0.3,stars(p),ha='center',va='bottom',fontsize=ANNOT_SIZE,color=ANNOT_COLOR,zorder=6)
    ax.text(i,ytop+3.0,f"{diff:+.1f}",ha='center',va='bottom',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i-w/2,ytop-3.2,f"{h.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i+w/2,ytop-3.2,f"{m.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=11)
ax.set_ylabel("Divergent thinking score", fontsize=11); ax.set_ylim(60,105)
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
leg.set_title(None)
# sample-size note directly under the legend box, left-aligned with it
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, f'n={SAMPLE_N} per group,\nrandom sampled', transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
ax.set_title("Panel A - Divergent Thinking Score by Split", fontsize=12, weight='bold')

ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0)
ax.set_axisbelow(True)
# exact 1:1 aspect per Dawei's snippet
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim()
ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelA.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print("PANEL A v4 (sampled + 1:1) DONE")


## Panel B code

In [ ]:
import csv, pickle, numpy as np, random
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w): 
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
Ht=top10(Hd); Mt=top10(Md)
# first 7 valid words per response
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in Ht]; Ha=[x for x in Ha if len(x)==7]
Ma=[first7(ws) for ws in Mt]; Ma=[x for x in Ma if len(x)==7]
# rarefaction capped at SAME 500 sampled responses per side
CAP=500; STEP=5; REPS=40
def rarefy(pop):
    # Between-respondent BOOTSTRAP: at each k, resample respondents WITH REPLACEMENT
    # from the full pool, then accumulate distinct words. Band = spread across
    # respondent compositions (real heterogeneity), not just draw stability.
    N=len(pop); n=min(CAP,N); xs=list(range(STEP,n+1,STEP)); ys=[];lo=[];hi=[]
    BREPS=300
    for k in xs:
        c=[]
        for _ in range(BREPS):
            idx=np.random.randint(0,N,size=k)  # with replacement
            s=set()
            for i in idx: s.update(pop[i])
            c.append(len(s))
        ys.append(np.mean(c));lo.append(np.percentile(c,2.5));hi.append(np.percentile(c,97.5))
    return xs,ys,lo,hi
hx,hy,hl,hh=rarefy(Ha); mx,my,ml,mh=rarefy(Ma)
HUMAN="#5E348B"; MACHINE="#3CB7B0"
fig,ax=plt.subplots(figsize=(7,7))
ax.plot(hx,hy,color=HUMAN,lw=2,label="Human"); ax.fill_between(hx,hl,hh,color=HUMAN,alpha=0.2)
ax.plot(mx,my,color=MACHINE,lw=2,label="Machine"); ax.fill_between(mx,ml,mh,color=MACHINE,alpha=0.2)
ax.set_xlabel("Sampled responses", fontsize=11); ax.set_ylabel("Cumulative distinct words", fontsize=11)
ax.set_title("Panel B - Cumulative Distinct Words", fontsize=12, weight='bold')
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='both', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0)
ax.set_axisbelow(True)
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, "top 10%, first 7 words\n500 resp., respondent bootstrap 95% CI", transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
# endpoint distinct-word counts, top-center-right
ax.text(0.66,0.97,"Distinct words @500:",transform=ax.transAxes,fontsize=9.5,color="#333",va="top",ha="left",weight="bold")
ax.text(0.66,0.915,f"Human  {int(round(hy[-1]))}",transform=ax.transAxes,fontsize=10,color=HUMAN,va="top",ha="left",weight="bold")
ax.text(0.66,0.865,f"Machine  {int(round(my[-1]))}",transform=ax.transAxes,fontsize=10,color=MACHINE,va="top",ha="left",weight="bold")
ax.set_xlim(0,CAP+5)
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim(); ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelB.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print(f"Panel B done. human@500={hy[-1]:.0f} machine@500={my[-1]:.0f} (Ha={len(Ha)} Ma={len(Ma)})")


## Panel C - Relative Word Frequency
Top 10%, first 7 valid words; n=500/group sampled; per-response mean Zipf frequency (lower = rarer). Welch t + Cohen's d.


In [ ]:
import csv, pickle, numpy as np, random
from scipy import stats
from wordfreq import zipf_frequency
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w):
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
Ht=top10(Hd); Mt=top10(Md)
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in Ht if len(first7(ws))==7]
Ma=[first7(ws) for ws in Mt if len(first7(ws))==7]
# sample n=500 responses per side
SAMPLE_N=500
random.shuffle(Ha); random.shuffle(Ma)
Ha=Ha[:SAMPLE_N]; Ma=Ma[:SAMPLE_N]
# per-response MEAN zipf frequency (lower = rarer)
def resp_zipf(pop): return [np.mean([zipf_frequency(w,'en') for w in a]) for a in pop]
hz=resp_zipf(Ha); mz=resp_zipf(Ma)
hmu,mmu=np.mean(hz),np.mean(mz)
t,p=stats.ttest_ind(hz,mz,equal_var=False)
d=(mmu-hmu)/np.sqrt((np.var(hz,ddof=1)+np.var(mz,ddof=1))/2)
HUMAN="#5E348B"; MACHINE="#3CB7B0"
fig,ax=plt.subplots(figsize=(7,7))
bins=np.linspace(min(min(hz),min(mz)),max(max(hz),max(mz)),34)
ax.hist(hz,bins=bins,color=HUMAN,alpha=0.55,density=True,label="Human",edgecolor='white',linewidth=0.3)
ax.hist(mz,bins=bins,color=MACHINE,alpha=0.55,density=True,label="Machine",edgecolor='white',linewidth=0.3)
ax.axvline(hmu,color=HUMAN,ls='--',lw=1.6); ax.axvline(mmu,color=MACHINE,ls='--',lw=1.6)
ax.set_xlabel("Mean word frequency (Zipf; lower = rarer)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Panel C - Relative Word Frequency", fontsize=12, weight='bold')
def stars(p): return "***" if p<1e-3 else "**" if p<1e-2 else "*" if p<0.05 else "ns"
ax.text(0.66,0.97,"Mean Zipf:",transform=ax.transAxes,fontsize=9.5,color="#333",va="top",ha="left",weight="bold")
ax.text(0.66,0.915,f"Human  {hmu:.2f}",transform=ax.transAxes,fontsize=10,color=HUMAN,va="top",ha="left",weight="bold")
ax.text(0.66,0.865,f"Machine  {mmu:.2f}",transform=ax.transAxes,fontsize=10,color=MACHINE,va="top",ha="left",weight="bold")
ax.text(0.66,0.815,f"diff {mmu-hmu:+.2f}  {stars(p)}",transform=ax.transAxes,fontsize=9.5,color="#333",va="top",ha="left")
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='both', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0); ax.set_axisbelow(True)
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, f"n={SAMPLE_N} per group,\nrandom sampled", transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim(); ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelC.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print(f"Panel C done. Human Zipf {hmu:.3f} vs Machine {mmu:.3f}, d={d:.2f} p={p:.1e}")
